<a href="https://colab.research.google.com/github/gundalarakesh262-cpu/nifty100-financial-intelligence/blob/main/notebooks/Investment_Screener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

ratios_path = "../data/processed/financial_ratios_generated.csv"

ratios = pd.read_csv(ratios_path)

print("Rows:", ratios.shape[0])
print("Columns:", ratios.shape[1])

ratios.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/financial_ratios_generated.csv'

In [11]:
# Make sure fiscal_year is numeric
ratios["fiscal_year"] = pd.to_numeric(ratios["fiscal_year"], errors="coerce")

# Remove invalid year rows
ratios_clean = ratios[ratios["fiscal_year"].notna()].copy()

# Sort and pick latest year per company
latest = (
    ratios_clean
    .sort_values(["company_id", "fiscal_year"])
    .groupby("company_id")
    .tail(1)
    .reset_index(drop=True)
)

print("Latest company universe:", latest.shape)
latest[["company_id", "year", "fiscal_year"]].head()

Latest company universe: (98, 62)


,company_id,year,fiscal_year
0,ABB,Mar 2024,2024
1,ADANIENSOL,Mar 2024,2024
2,ADANIENT,Mar 2024,2024
3,ADANIGREEN,Mar 2024,2024
4,ADANIPORTS,Mar 2024,2024


In [12]:
def percentile_score(series, higher_is_better=True):
    """
    Converts a numeric column into 0-100 percentile score.
    """
    s = pd.to_numeric(series, errors="coerce")

    if higher_is_better:
        return s.rank(pct=True) * 100
    else:
        return (1 - s.rank(pct=True)) * 100


scored = latest.copy()

# Profitability score
scored["score_roe"] = percentile_score(scored["return_on_equity_pct"], higher_is_better=True)
scored["score_roce"] = percentile_score(scored["return_on_capital_employed_pct"], higher_is_better=True)
scored["score_npm"] = percentile_score(scored["net_profit_margin_pct"], higher_is_better=True)

# Growth score
scored["score_revenue_growth"] = percentile_score(scored["revenue_cagr_5y_pct"], higher_is_better=True)
scored["score_pat_growth"] = percentile_score(scored["pat_cagr_5y_pct"], higher_is_better=True)

# Debt score: lower debt is better
scored["score_debt"] = percentile_score(scored["debt_to_equity"], higher_is_better=False)

# Cash flow score
scored["score_fcf"] = percentile_score(scored["free_cash_flow_cr"], higher_is_better=True)
scored["score_fcf_yield"] = percentile_score(scored["fcf_yield_pct"], higher_is_better=True)

# Valuation score: lower P/E and P/B are better
scored["score_pe"] = percentile_score(scored["pe_ratio"], higher_is_better=False)
scored["score_pb"] = percentile_score(scored["pb_ratio"], higher_is_better=False)

# Composite score
scored["composite_score"] = (
    scored["score_roe"].fillna(50) * 0.15 +
    scored["score_roce"].fillna(50) * 0.15 +
    scored["score_npm"].fillna(50) * 0.10 +
    scored["score_revenue_growth"].fillna(50) * 0.15 +
    scored["score_pat_growth"].fillna(50) * 0.15 +
    scored["score_debt"].fillna(50) * 0.10 +
    scored["score_fcf"].fillna(50) * 0.10 +
    scored["score_pe"].fillna(50) * 0.05 +
    scored["score_pb"].fillna(50) * 0.05
)

scored = scored.sort_values("composite_score", ascending=False)

scored[[
    "company_id",
    "fiscal_year",
    "return_on_equity_pct",
    "return_on_capital_employed_pct",
    "debt_to_equity",
    "free_cash_flow_cr",
    "revenue_cagr_5y_pct",
    "pat_cagr_5y_pct",
    "pe_ratio",
    "pb_ratio",
    "composite_score"
]].head(20)

,company_id,fiscal_year,return_on_equity_pct,return_on_capital_employed_pct,debt_to_equity,free_cash_flow_cr,revenue_cagr_5y_pct,pat_cagr_5y_pct,pe_ratio,pb_ratio,composite_score
46,INDIGO,2024,892.568306,4953.540773,0.018579,9424.0,19.313355,120.692509,76.23,7.87,79.974105
50,IRCTC,2024,34.396285,42.826748,0.018576,667.0,17.955244,29.166864,60.44,5.74,75.169912
89,TRENT,2024,36.307768,22.332932,0.430924,841.0,36.306916,73.113676,8.65,11.82,71.347710
61,LTIM,2024,22.904386,25.207117,0.103457,1752.0,30.327981,24.775129,64.52,2.38,70.609001
35,HAL,2024,3816.582915,2590.993789,0.618090,1814.0,8.712549,26.485271,71.89,3.65,69.373699
5,ADANIPOWER,2024,48.276741,18.385823,0.802318,17651.0,16.086096,NaN,44.57,9.91,66.668946
48,INFY,2024,29.788007,32.906971,0.094864,20117.0,13.199101,11.239420,35.99,8.97,65.773503
66,NESTLEIND,2024,117.754491,143.147897,0.103293,2938.0,14.548574,14.852320,54.96,14.34,65.391557
58,LICI,2024,49.447110,39.050358,0.000000,853.0,8.159861,73.175567,30.67,8.10,65.258919
52,ITC,2024,27.851074,32.638685,0.004067,18742.0,7.950897,10.083408,47.85,3.24,65.204650


In [14]:
def run_screener(
    data,
    min_roe=None,
    min_roce=None,
    max_debt_to_equity=None,
    min_fcf=None,
    min_revenue_cagr_5y=None,
    min_pat_cagr_5y=None,
    max_pe=None,
    max_pb=None,
    min_net_profit_margin=None,
    debt_free_only=False,
    fcf_positive_only=False,
    min_composite_score=None,
    top_n=50
):
    result = data.copy()

    if min_roe is not None:
        result = result[result["return_on_equity_pct"] >= min_roe]

    if min_roce is not None:
        result = result[result["return_on_capital_employed_pct"] >= min_roce]

    if max_debt_to_equity is not None:
        result = result[result["debt_to_equity"] <= max_debt_to_equity]

    if min_fcf is not None:
        result = result[result["free_cash_flow_cr"] >= min_fcf]

    if min_revenue_cagr_5y is not None:
        result = result[result["revenue_cagr_5y_pct"] >= min_revenue_cagr_5y]

    if min_pat_cagr_5y is not None:
        result = result[result["pat_cagr_5y_pct"] >= min_pat_cagr_5y]

    if max_pe is not None:
        result = result[result["pe_ratio"] <= max_pe]

    if max_pb is not None:
        result = result[result["pb_ratio"] <= max_pb]

    if min_net_profit_margin is not None:
        result = result[result["net_profit_margin_pct"] >= min_net_profit_margin]

    if debt_free_only:
        result = result[result["debt_free_flag"] == True]

    if fcf_positive_only:
        result = result[result["fcf_positive_flag"] == True]

    if min_composite_score is not None:
        result = result[result["composite_score"] >= min_composite_score]

    result = result.sort_values("composite_score", ascending=False)

    return result.head(top_n)

In [15]:
preset_results = {}

# 1. Quality Companies
preset_results["quality"] = run_screener(
    scored,
    min_roe=15,
    min_roce=15,
    max_debt_to_equity=1,
    min_fcf=0,
    top_n=50
)

# 2. Value Companies
preset_results["value"] = run_screener(
    scored,
    max_pe=25,
    max_pb=4,
    min_roe=10,
    top_n=50
)

# 3. Growth Companies
preset_results["growth"] = run_screener(
    scored,
    min_revenue_cagr_5y=10,
    min_pat_cagr_5y=10,
    min_roe=10,
    top_n=50
)

# 4. Debt-Free Companies
preset_results["debt_free"] = run_screener(
    scored,
    debt_free_only=True,
    min_roe=10,
    fcf_positive_only=True,
    top_n=50
)

# 5. Cash Flow Kings
preset_results["cash_flow_kings"] = run_screener(
    scored,
    min_fcf=0,
    fcf_positive_only=True,
    min_net_profit_margin=10,
    top_n=50
)

# 6. Balanced Winners
preset_results["balanced_winners"] = run_screener(
    scored,
    min_roe=12,
    min_roce=12,
    max_debt_to_equity=1.5,
    min_revenue_cagr_5y=8,
    min_composite_score=60,
    top_n=50
)

for name, result in preset_results.items():
    print(name, ":", len(result), "companies")

quality : 36 companies
value : 5 companies
growth : 41 companies
debt_free : 1 companies
cash_flow_kings : 43 companies
balanced_winners : 18 companies


In [17]:
display_cols = [
    "company_id",
    "year",
    "fiscal_year",
    "composite_score",
    "return_on_equity_pct",
    "return_on_capital_employed_pct",
    "net_profit_margin_pct",
    "debt_to_equity",
    "free_cash_flow_cr",
    "revenue_cagr_5y_pct",
    "pat_cagr_5y_pct",
    "pe_ratio",
    "pb_ratio",
    "capital_allocation_pattern"
]

preset_results["quality"][display_cols].head(20)

,company_id,year,fiscal_year,composite_score,return_on_equity_pct,return_on_capital_employed_pct,net_profit_margin_pct,debt_to_equity,free_cash_flow_cr,revenue_cagr_5y_pct,pat_cagr_5y_pct,pe_ratio,pb_ratio,capital_allocation_pattern
46,INDIGO,Mar 2024,2024,79.974105,892.568306,4953.540773,11.852723,0.018579,9424.0,19.313355,120.692509,76.23,7.87,Reinvesting + Returning Capital
50,IRCTC,Mar 2024,2024,75.169912,34.396285,42.826748,26.018735,0.018576,667.0,17.955244,29.166864,60.44,5.74,Reinvesting + Returning Capital
89,TRENT,Mar 2024,2024,71.347710,36.307768,22.332932,11.935354,0.430924,841.0,36.306916,73.113676,8.65,11.82,Reinvesting + Returning Capital
61,LTIM,Mar 2024,2024,70.609001,22.904386,25.207117,12.909311,0.103457,1752.0,30.327981,24.775129,64.52,2.38,Reinvesting + Returning Capital
35,HAL,Mar 2024,2024,69.373699,3816.582915,2590.993789,24.999177,0.618090,1814.0,8.712549,26.485271,71.89,3.65,Reinvesting + Returning Capital
5,ADANIPOWER,Mar 2024,2024,66.668946,48.276741,18.385823,41.367599,0.802318,17651.0,16.086096,NaN,44.57,9.91,Asset Sale + Debt/Dividend Outflow
48,INFY,Mar 2024,2024,65.773503,29.788007,32.906971,17.080757,0.094864,20117.0,13.199101,11.239420,35.99,8.97,Reinvesting + Returning Capital
66,NESTLEIND,Mar 2024,2024,65.391557,117.754491,143.147897,16.122817,0.103293,2938.0,14.548574,14.852320,54.96,14.34,Reinvesting + Returning Capital
58,LICI,Mar 2024,2024,65.258919,49.447110,39.050358,4.836601,0.000000,853.0,8.159861,73.175567,30.67,8.10,Reinvesting + Returning Capital
52,ITC,Mar 2024,2024,65.204650,27.851074,32.638685,29.282025,0.004067,18742.0,7.950897,10.083408,47.85,3.24,Asset Sale + Debt/Dividend Outflow


In [18]:
os.makedirs("../data/processed", exist_ok=True)

# Main screener output: all companies ranked
scored.to_csv("../data/processed/screener_full_ranked_universe.csv", index=False)

# Summary of preset counts
preset_summary = pd.DataFrame({
    "preset_name": list(preset_results.keys()),
    "company_count": [len(df) for df in preset_results.values()]
})

preset_summary.to_csv("../data/processed/screener_presets_summary.csv", index=False)

# Excel output with one sheet per preset
excel_path = "../data/processed/screener_output.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    scored[display_cols].to_excel(writer, sheet_name="Full Ranked Universe", index=False)

    for name, result in preset_results.items():
        result[display_cols].to_excel(writer, sheet_name=name[:31], index=False)

print("Screener output files created successfully!")
print("Saved:", excel_path)

preset_summary

Screener output files created successfully!
Saved: ../data/processed/screener_output.xlsx


,preset_name,company_count
0,quality,36
1,value,5
2,growth,41
3,debt_free,1
4,cash_flow_kings,43
5,balanced_winners,18
